### Mini Tutorial: Blind LRG1 Power Spectrum and Bispectrum Measurements

This is a short tutorial on how to blind your clustering using the `desiblind` API. We will load the measurements using the `desi-clustering` `full_shape` package for convenience.


In [3]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from full_shape import tools
from desiblind import (
    TracerBispectrumMultipolesBlinder,
    TracerPowerSpectrumMultipolesBlinder,
)

%matplotlib inline
%config InlineBackend.figure_format = "retina"


In [ ]:
STATS_DIR = Path(
    "/global/cfs/cdirs/desicollab/science/cai/desi-clustering/dr2/summary_statistics/full_shape/base"
)
DATASET = "abacus-2ndgen-dr2-complete"
TRACER = "LRG1"
COVARIANCE = "holi-v3-altmtl"


def load_measurement(stat):
    likelihood_options = tools.generate_likelihood_options_helper(
        stats=[stat],
        tracer=TRACER,
        version=DATASET,
        covariance=COVARIANCE,
        stats_dir=STATS_DIR,
    )
    stats = tools.get_stats(
        observables_options=likelihood_options["observables"],
        covariance_options=likelihood_options["covariance"],
    )
    data, windows, covariance = tools.unpack_stats(stats)
    return data[0]


### Power Spectrum


In [ ]:
pk = load_measurement("mesh2_spectrum")

blinded_pk = TracerPowerSpectrumMultipolesBlinder.apply_blinding(
    name=TRACER,
    data=pk,
)

fig, ax = plt.subplots(figsize=(5, 4))
for ell in blinded_pk.ells:
    pole = blinded_pk.get(ells=ell)
    k = pole.coords("k")
    ax.plot(k, k * pole.value(), label=rf"$\ell={ell}$")
ax.set_xlabel(r"$k\,[h\,{\rm Mpc}^{-1}]$")
ax.set_ylabel(r"$k\,P_\ell(k)$")
ax.legend()
fig.tight_layout()
fig.show()


### Bispectrum


In [ ]:
bk = load_measurement("mesh3_spectrum")

blinded_bk = TracerBispectrumMultipolesBlinder.apply_blinding(
    name=TRACER,
    data=bk,
)

fig, ax = plt.subplots(figsize=(5, 4))
for ell in blinded_bk.ells:
    pole = blinded_bk.get(ells=ell)
    k = pole.coords("k")
    x = k[..., 0]
    ax.plot(x, k.prod(axis=-1) * pole.value(), label=str(ell))
ax.set_xlabel(r"$k\,[h\,{\rm Mpc}^{-1}]$")
ax.set_ylabel(r"$k^2 B_{\ell}(k, k)$ [$(\mathrm{Mpc}/h)^4$]")
ax.legend()
fig.tight_layout()
fig.show()


## Notes

- Bispectrum blinding currently supports only diagonal Sugiyama basis multipoles. In that case the shifts are interpolated in the diagonal `k` coordinate with the same spline behavior used for the power spectrum.
